# Make Odin dark, open beam and sample data from Ymir data

This notebook creates a file for Odin that contain images in the Orca detector file entry.
We use a ODIN file written by ECDC as a template and replace the orca detector data with images recorded at Ymir.

The Ymir data was a single file containing dark frame, open beam, and sample data,
with a flag indicating which frame is part of which recording.

We also generate a fake proton-charge log and place it inside the `neutron_prod_info` group.
The proton charge is a time series from the first frame time to the last frame time + exposure time, where the value is randomly oscillating around 1.

In [ ]:
import h5py as h5
import numpy as np
import scipp as sc

In [ ]:
from ess.ymir.data import ymir_lego_images_path

## Load Ymir data

In [ ]:
path = 'entry/instrument/histogram_mode_detectors/orca/'

with h5.File(ymir_lego_images_path()) as f:
    lego_data = f[path + "data/value"][()]
    lego_time = f[path + 'data/time'][()]
    imkey = f[path + 'image_key']
    image_key = {
        k: {'values': imkey[k][()], 'attrs': dict(imkey[k].attrs)} for k in imkey.keys()
    }

## Create fake proton charge

In [ ]:
exposure_time = int(5e8)  # 0.5s (made up)
period = 1e9 / 14

# Need to start a bit before the start of the time axis of the data
# so that sc.lookup in 'previous' mode can find the pulse time
charge_time = np.arange(
    float(lego_time[0]) - exposure_time, float(lego_time[-1] + exposure_time), period
).astype('uint64')
charge_value = np.random.uniform(0.99, 1.01, size=len(charge_time)).astype('float64')

## Make pixel offsets

In [ ]:
npix = lego_data.shape[-1]

x_pixel_offset = sc.linspace('dim_1', -0.1, 0.1, npix, unit='m')
y_pixel_offset = sc.linspace('dim_2', -0.1, 0.1, npix, unit='m')
x_pixel_offset

## Create new files from template

In [ ]:
def replace_dataset(entry, name, values):
    attrs = dict(entry[name].attrs)
    del entry[name]
    dset = entry.create_dataset(name, data=values)
    dset.attrs.update(attrs)


def make_new_file(template_nexus_file, outfile, data, proton_charge, image_key):
    import shutil

    shutil.copyfile(template_nexus_file, outfile)

    instr_path = 'entry/instrument/'

    detector_path = instr_path + 'histogram_mode_detectors/orca/'  # ODIN
    # detector_path = instr_path + 'orca_detector/'  # TBL
    proton_charge_path = 'entry/neutron_prod_info/pulse_charge'

    with h5.File(outfile, "r+") as f:
        detector_data = f[detector_path + "data"]
        replace_dataset(detector_data, name="value", values=data["value"])
        replace_dataset(detector_data, name="time", values=data["time"])
        detector_data.attrs['axes'] = ['time', 'y_pixel_offset', 'x_pixel_offset']

        detector = f[detector_path]

        imkey_attrs = dict(detector['image_key'].attrs)
        del detector['image_key']
        g = detector.create_group("image_key")
        g.attrs.update(imkey_attrs)
        for key, value in image_key.items():
            new = detector['image_key'].create_dataset(key, data=value['values'])
            new.attrs.update(value['attrs'])

        del detector['x_pixel_offset']
        xoff = detector.create_dataset('x_pixel_offset', data=x_pixel_offset.values)
        xoff.attrs['units'] = str(x_pixel_offset.unit)
        xoff.attrs['axis'] = 1
        del detector['y_pixel_offset']
        yoff = detector.create_dataset('y_pixel_offset', data=y_pixel_offset.values)
        yoff.attrs['units'] = str(y_pixel_offset.unit)
        yoff.attrs['axis'] = 2

        pcharge = f[proton_charge_path]
        replace_dataset(pcharge, name="value", values=proton_charge["value"])
        replace_dataset(pcharge, name="time", values=proton_charge["time"])
        replace_dataset(
            pcharge, name="average_value", values=proton_charge["value"].mean()
        )
        replace_dataset(
            pcharge, name="minimum_value", values=proton_charge["value"].min()
        )
        replace_dataset(
            pcharge, name="maximum_value", values=proton_charge["value"].max()
        )

        del f[instr_path + 'event_mode_detectors']

In [ ]:
template_nexus_file = "coda_odin_999999_00011093.hdf"

In [ ]:
# Remember to 'h5repack' the new file after it is created

make_new_file(
    template_nexus_file=template_nexus_file,
    outfile="ymir_lego_odin.hdf",
    data={"value": lego_data, "time": lego_time},
    proton_charge={"value": charge_value, "time": charge_time},
    image_key=image_key,
)